# XLM-RoBERTa Model Eğitimi - Google Colab

Bu notebook, Google Colab'da ücretsiz GPU kullanarak XLM-RoBERTa modeli eğitir.

## Adımlar:
1. Bu notebook'u Google Colab'a yükleyin
2. GPU'yu aktif edin (Runtime > Change runtime type > GPU)
3. Dataset'i yükleyin
4. Tüm hücreleri çalıştırın
5. Eğitilmiş modeli indirin

## 1. Gerekli Paketleri Yükle

In [ ]:
!pip install transformers datasets accelerate torch scikit-learn numpy -q

## 2. Dataset'i Yükle

Projenizdeki dataset dosyalarını buraya yükleyin:
- Colab'a zip olarak yükleyin
- Veya Google Drive'dan bağlayın
- Veya GitHub'dan clone edin

In [ ]:
# Yöntem 1: Google Drive bağla (önerilen)
from google.colab import drive
drive.mount('/content/drive')

# Dataset yolunu ayarlayın
# Örnek: DATASET_PATH = '/content/drive/MyDrive/graduation_project/ml-service/inference/services/text_analyzer/data'

In [ ]:
# Yöntem 2: GitHub'dan clone (eğer repo public ise)
# !git clone https://github.com/yourusername/graduation_project.git
# DATASET_PATH = '/content/graduation_project/ml-service/inference/services/text_analyzer/data'

In [ ]:
# Yöntem 3: Zip dosyası yükle
# from google.colab import files
# uploaded = files.upload()  # data.zip yükleyin
# !unzip data.zip
# DATASET_PATH = '/content/data'

## 3. Data Processor'ı Yükle

`data_processor.py` dosyasını buraya kopyalayın veya yükleyin

In [ ]:
# data_processor.py dosyasını buraya yapıştırın veya yükleyin
# Alternatif: Dosyayı Colab'a yükleyin
# from google.colab import files
# files.upload()  # data_processor.py seçin

## 4. XLM-RoBERTa Eğitim Fonksiyonu

In [ ]:
import sys
import json
from pathlib import Path
from collections import Counter

import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from datasets import Dataset as HFDataset

# GPU kontrolü
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
def compute_metrics(eval_pred):
    """Metrics hesaplama fonksiyonu"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average='binary', zero_division=0)
    recall = recall_score(labels, predictions, average='binary', zero_division=0)
    f1 = f1_score(labels, predictions, average='binary', zero_division=0)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [ ]:
# Dataset yükleme
# data_processor.py'yi import edin veya burada tanımlayın
from data_processor import DataProcessor

print("Dataset yukleniyor...")
processor = DataProcessor(data_dir=DATASET_PATH)  # DATASET_PATH'ı yukarıda tanımlayın
texts, labels, sources = processor.process_all_datasets()

print(f"Toplam kayit: {len(texts)}")
print(f"Disaster related: {sum(labels)} ({sum(labels)/len(labels)*100:.1f}%)")
print(f"Not related: {len(labels) - sum(labels)} ({(len(labels) - sum(labels))/len(labels)*100:.1f}%)")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test, sources_train, sources_test = train_test_split(
    texts, labels, sources, test_size=0.2, random_state=42, stratify=labels
)

# Validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

print(f"Train set: {len(X_train)} kayit")
print(f"Validation set: {len(X_val)} kayit")
print(f"Test set: {len(X_test)} kayit")

In [ ]:
# Tokenizer ve Model yükleme
MODEL_NAME = "xlm-roberta-base"  # veya "xlm-roberta-large"

print(f"Tokenizer yukleniyor: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Model yukleniyor: {MODEL_NAME}")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    problem_type="single_label_classification"
)
model.to(device)

In [ ]:
# Dataset hazırlama
MAX_LENGTH = 512

train_dataset = HFDataset.from_dict({'text': X_train, 'label': y_train})
val_dataset = HFDataset.from_dict({'text': X_val, 'label': y_val})
test_dataset = HFDataset.from_dict({'text': X_test, 'label': y_test})

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column('label', 'labels')
val_dataset = val_dataset.rename_column('label', 'labels')
test_dataset = test_dataset.rename_column('label', 'labels')

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print("Datasets hazir!")

In [ ]:
# Training arguments
OUTPUT_DIR = "/content/xlm_roberta_model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=500,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=3,
    fp16=torch.cuda.is_available(),
    report_to="none"  # TensorBoard'u kapat
)

In [ ]:
# Trainer oluştur ve eğit
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Egitim basliyor...")
train_result = trainer.train()

# Modeli kaydet
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nModel kaydedildi: {OUTPUT_DIR}")

In [ ]:
# Test set evaluation
print("\nTest set evaluation...")
test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=1)
test_labels = test_predictions.label_ids

test_accuracy = accuracy_score(test_labels, test_preds)
test_precision = precision_score(test_labels, test_preds, average='binary', zero_division=0)
test_recall = recall_score(test_labels, test_preds, average='binary', zero_division=0)
test_f1 = f1_score(test_labels, test_preds, average='binary', zero_division=0)

print(f"\nTest Set Metrics:")
print(f"  Accuracy:  {test_accuracy:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall:    {test_recall:.4f}")
print(f"  F1-Score:  {test_f1:.4f}")

## 5. Modeli İndir

Eğitilmiş modeli bilgisayarınıza indirin

In [ ]:
# Modeli zip olarak paketle
!cd /content && zip -r xlm_roberta_model.zip xlm_roberta_model/

# İndir
from google.colab import files
files.download('/content/xlm_roberta_model.zip')

print("\nModel indirildi! Projenizdeki models/xlm_roberta/ klasörüne çıkartın.")

## Alternatif: Google Drive'a Kaydet

Modeli Drive'a kaydedip daha sonra indirebilirsiniz

In [ ]:
# Google Drive'a kopyala
# !cp -r /content/xlm_roberta_model /content/drive/MyDrive/
# print("Model Google Drive'a kaydedildi!")